In [ ]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")

In [ ]:
experiment_file = '../../experiments/parallelized_experiments/output/cosine_curve_amplitude_full_period/2023_12_17_15_52//experiment_result.json'

stiffness_path = '../../experiments/parallelized_experiments/output/cosine_curve_amplitude_full_period/2023_12_17_15_52/'

### Overview

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [ ]:
df = pd.DataFrame(data['data'])
fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 5))
a = (df.hist('Ipu simulation succeed', ax = axes[0]), df.hist('Planar equilibrium', ax = axes[1]), df.hist('Simulation Kappa value', ax = axes[2]))

In [ ]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)

In [ ]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [ ]:
kappa_path = None

In [ ]:
name = 'cosine_curve_amplitude_full_period'

In [ ]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [ ]:
parameters = (np.array(data['pattern_parameters'][0]['values']))

In [ ]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [ ]:
len(max_bending_stiffness)

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [ ]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [ ]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

In [ ]:
parameters

In [ ]:
parameters[(np.where(min_bending_stiffness > 0.1))]

In [ ]:
parameters[(np.where(min_stretching_stiffness > 4))]

### Get scale function convex hull

In [ ]:
import matplotlib.cm as cm
import matplotlib as mpl

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [ ]:
hull

In [ ]:
import matplotlib.pyplot as plt
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')
plt.plot(points[hull.vertices,0], points[hull.vertices,1], 'r--', lw=2)
plt.plot(points[hull.vertices[0],0], points[hull.vertices[0],1], 'ro')
plt.show()

In [ ]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

# plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)


points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

ax.title.set_text("Scale factors")
plt.xlabel("x scale factors")
plt.ylabel("y scale factors")

plt.scatter(max_scale_factors, min_scale_factors, label = 'min_stiffness', s = 200, alpha = 1, c = min_bending_stiffness)
# plt.scatter(x_scale_factors, y_scale_factors, label = 'max_stiffness', s = 200, alpha = 1, c = max_bending_stiffness)

# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)

### Validate the max and min scale factors are aligned with the x and y axis

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
eqns = hull.equations

In [ ]:
hull.max_bound, hull.min_bound

In [ ]:
import parametrization_helper, importlib
importlib.reload(parametrization_helper)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [ ]:
# for i in range(5):
#     for j in range(30):
#         stiffness_coefficients[:, i] = parametrization_helper.savitzky_golay(stiffness_coefficients[:, i], 11, 3) # window size 51, polynomial order 3

In [ ]:
plt.plot(stiffness_coefficients[:, 4])

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
np.argmax(stiffness_coefficients[:, 1]), np.argmax(stiffness_coefficients[:, 2])

In [ ]:
grid_data = np.zeros((9, len(parameters)))

In [ ]:
for i in range(len(parameters)):
    grid_data[0][i] = max_scale_factors[i]
    grid_data[1][i] = min_scale_factors[i]
    grid_data[2][i] = x_scale_factors[i]
    grid_data[3][i] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][i] = stiffness_coefficients[i][s]

In [ ]:
# np.save("grid_pattern_1.npy", grid_pattern_1)
# np.save("grid_pattern_2.npy", grid_pattern_2)
# np.save("grid_data.npy", grid_data)

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, (parameters))

In [ ]:
grid_data.shape

In [ ]:
scale_factors_grid_data = np.zeros((2, len(parameters)))
for i in range(len(parameters)):
    scale_factors_grid_data[0][i] = x_scale_factors[i]
    scale_factors_grid_data[1][i] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, (parameters))

In [ ]:
test_parameters = np.linspace(0, 0.9, 100)

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
# titles = ['max scale factors', 'min scale factors', 's1', 's2', 's3', 's4', 's5']

for i in range(9):
    axes[i].plot(test_parameters, splines[i * 3 + 1](test_parameters))
    axes[i].set_title(titles[i], fontsize=21)

In [ ]:
stiffness_coefficients = np.array(stiffness_coefficients)

In [ ]:
stiffness_coefficients.shape

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
data = [max_scale_factors, min_scale_factors, x_scale_factors, y_scale_factors, stiffness_coefficients[:, 0], stiffness_coefficients[:, 1], stiffness_coefficients[:, 2], stiffness_coefficients[:, 3], stiffness_coefficients[:, 4]]

for i in range(9):
    axes[i].plot(parameters, data[i])
    axes[i].set_title(titles[i], fontsize=21)

### End data generating

### Parametrization

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
lines = np.array(eqns)

### SIGGRAPH 21 Local global

In [ ]:
# Run some iterations of the local-global algorithm to ensure a good separation between singular values.
# This step can also be used as a prediction of the feasiblity of a design surface:
# if it is unable to nearly satisfy the singular value constraints,
# the surface is probably infeasible.
lg_21 = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

lg_21.alphaMin = 1.4
lg_21.alphaMax = np.pi / 2
print(lg_21.energy())
for i in range(1000): lg_21.runIteration()

print(lg_21.energy())
lg_21.runIteration()
print(lg_21.energy())

In [ ]:
visualization.visualize(lg_21)

In [ ]:
new_lines = np.array([[0, 1, -1.05], [1, 0, -np.pi / 2.], [0, -1, 0.95], [-1., 0, 1.4]])
             # , [0, -1, -1.1], [1, 0, -1.4], [-1, 0, np.pi / 2]]

In [ ]:
parametrization_helper.visualize_scale_factors(new_lines, lg_21.getAlphas(), np.ones(len(lg_21.getAlphas())), )

### New local global to compare

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

lg.betaMin = 1.0
lg.betaMax = 1.0

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg)

### New local global with convex hull

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)

lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [ ]:
visualization.visualize_both(lg)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [ ]:
default_pattern_params = [0.]  * len(lg.getAlphas())

In [ ]:
mat_info = np.array(default_pattern_params).reshape((1, len(lg.getAlphas())))

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.0, 0.0]])
rparam.diffRegW = 0.0

In [ ]:
visualization.visualize_both(rparam, height = 4, showBarriers=True)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.bendRegW = 1

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.RGP, PET.Bending]))

In [ ]:
def optimize_rparam(param, patternRegW, phiRegW, bendRegW = 0.0, update_uv = True, niter = 100):
    param.patternRegW = patternRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = niter
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    if update_uv:
        fixedvars = [param.uOffset(), param.vOffset(), param.phiOffset()]
    else:
        fixedvars = range(param.stretchOffset())

    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, fixedvars)
    benchmark.report()
    return cr

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4, showBarriers=True)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False, num_pattern_vars=1)

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 0, bendRegW = 0, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4, showBarriers=True)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False, num_pattern_vars=1)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
# no_regularization_var = rparam.getVars()

In [ ]:
# two_separate_optimization_vars = rparam.getVars()

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4, showBarriers=True)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False, num_pattern_vars=1)

In [ ]:
rparam.bendRegW = 1e-3
rparam.patternRegW = 1e-3
rparam.phiRegW = 1e-5
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-3, phiRegW = 1e-5, bendRegW = 1e-3, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
rparam.bendRegW = 1e-3
rparam.patternRegW = 1e-3
rparam.phiRegW = 1e-5
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4, showBarriers=True)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False, num_pattern_vars=1)

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-3, phiRegW = 1e-5, bendRegW = 1e-3, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-5, bendRegW = 1e1, update_uv = True, niter = 200)
benchmark.report()

In [ ]:

PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4, showBarriers=True)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False, num_pattern_vars=1)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 5, height = 5)

In [ ]:
visualization.visualizeChannelOrientationWithIsotropicPoints(rparam, quiver=visualization.QuiverVisualization.PER_TRI, orientationHue=False, width = 5, height = 5, use_x_axis=True)

In [ ]:
importlib.reload(visualization)
importlib.reload(parametrization_helper)

In [ ]:
rparam.get_stretch_angle_offset_from_pattern_params(rparam.getMatInfoArgs())

In [ ]:
max(parametrization_helper.get_stretch_angle_offset_from_pattern_params(scale_factors_splines, rparam.getPatternParams().reshape(1, -1)))

In [ ]:
rparam.get_stretch_angle_offset_from_pattern_params(rparam.getPatternParams().reshape(1, -1))

### Validation

In [ ]:
# import fd_validation
# # fd_validation.validateGrad(rsvd, fd_eps=1e-8,
# #                            xeval=rsvd.getVars() + 1e-3 * np.random.uniform(low=-1, high=1, size=rsvd.numVars()),
# #                            perturb=perturb[0:rsvd.numVars()], fixedVars=fixedVars)

# rparam.energy(energyType = rparam.PatternEnergyType.Full)

# perturb = np.random.uniform(low = -1, high = 1, size = rparam.numVars())

# perturb[rparam.stretchOffset():] *= 0
# # perturb[:rsvd.rgp.stretchOffset()] *= 0


# fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.RGP}, perturb = perturb)

# fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.RGP}, epsilons = np.logspace(-9, -1, 100))

# fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.Bending}, epsilons = np.logspace(-9, -1, 100))

# fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.PatternRegularization})

# fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.Full})

# fd_validation.hessConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.Bending})

# fd_validation.hessConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.RGP})

In [ ]:
# var_types = ['beforeStretching', 'p1']
# var_indices = {'beforeStretching': np.arange(rparam.stretchOffset()),
#                'p1': np.arange(rparam.stretchOffset(), rparam.stretchOffset() + len(lg.getAlphas()), 1)}

In [ ]:
# fd_validation.hessian_convergence_block_plot(rparam, var_types, var_indices, customArgs = {"energyType": rparam.PatternEnergyType.RGP})

In [ ]:
# fd_validation.hessian_convergence_block_plot(rparam, var_types, var_indices, customArgs = {"energyType": rparam.PatternEnergyType.Bending})

## Upsampling and channel generation

In [ ]:
nsubdiv=4
upsampledMesh, upsampledAngles, upsampledPatternParams = rparam.upsampledVertexLeftStretchAnglesAndPatternParameters(nsubdiv)
upsampleMesh_vertices = upsampledMesh.vertices()
upsampleMesh_triangles = upsampledMesh.triangles()
angle_offset = parametrization_helper.get_stretch_angle_offset_from_pattern_params(scale_factors_splines, np.array(upsampledPatternParams).reshape((1, -1)))
upsampledAngles += angle_offset

In [ ]:
# min(angle_offset), max(angle_offset)

In [ ]:
np.save("upsampleMesh_vertices.npy", upsampledMesh.vertices())
np.save("upsampleMesh_triangles.npy", upsampledMesh.triangles())
np.save("upsampleAngles.npy", upsampledAngles)
np.save("upsampledPatternParams.npy", upsampledPatternParams)

In [ ]:
upsampleMesh_vertices = np.load("upsampleMesh_vertices.npy")
upsampleMesh_triangles = np.load("upsampleMesh_triangles.npy")
upsampleAngles = np.load("upsampleAngles.npy")
upsampledPatternParams = np.load("upsampledPatternParams.npy")

In [ ]:
np.set_printoptions(suppress=True)

In [ ]:
max(upsampledPatternParams[0])

In [ ]:
amp_data =  upsampledPatternParams[0]


In [ ]:
import igl

In [ ]:
from parametrization_helper import get_distance_to_line_segments

In [ ]:
def fusing_curve_polyline(patternParams):
#     Draw cosine curves.
    amp = patternParams[0]
    def get_y_from_x(x):
        return amp * np.cos(x) * 0.5 * np.pi + np.pi / 2
    
    x_coords = np.linspace(-np.pi, np.pi, 10)
    y_coords = get_y_from_x(x_coords)
    x_coords += np.pi
    x_coords /= 2
    polyline = np.concatenate(((y_coords).reshape(-1, 1), (x_coords).reshape(-1, 1)), axis = 1)
    return polyline

In [ ]:
def pattern_function(theta, gamma, patternParams, margin, draw_boundary = False):
    if draw_boundary:
#         This is for debugging only and shouldn't be used for generating the inflatable mesh.
        if (theta < 0.1):
            return - margin
        if (gamma < 0.1):
            return - margin
    # Gamma is y, theta is x
    # Gamma theta are between 0 and pi
    polyline = fusing_curve_polyline(patternParams)    
    polyline_dist = get_distance_to_line_segments(np.array([theta, gamma]), polyline)
    return polyline_dist - margin

In [ ]:
pattern_function(np.pi / 4, np.pi / 10, [0.], 0.)

In [ ]:
# upsampledPatternParams = np.ones_like(upsampledPatternParams) * 0.4

In [ ]:
import time
start_time = time.time()
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_cross_field_custom_pattern(upsampleMesh_vertices, upsampleMesh_triangles, upsampleAngles, upsampledPatternParams, pattern_function, frequency=0.1, margin = 0.07)
print(time.time() - start_time)

# pickle.dump((sdfVertices, sdfTris, sdf), open('stripe_sdf_ns4_f100.pkl', 'wb'))

# import pickle, mesh, wall_generation, visualization, numpy as np
# (sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns4_f100.pkl', 'rb'))

In [ ]:
importlib.reload(visualization)

import matplotlib as mpl

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 10, height=10)

In [ ]:
importlib.reload(visualization)
visualization.scalarFieldPlotZeroContourFast(sdfVertices, sdfTris, sdf, width = 15, height=10, cmap = mpl.colormaps["PiYG"])

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=4.0,
                                              minContourLen=10)

visualization.plot_line_segments(pts, edges, width=10, height=10)

## Meshing and inflation simulation

In [ ]:
import sheet_meshing, inflation


In [ ]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(iwv) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(iwv) != 0)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 1e-2

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_parallel_tube_2023_12_17_low_frequency_without_bending.pkl.gz", 'wb'))

### Generate Fabrication Files

In [ ]:
scaleFactor = 1 # Factor for fine-tuning size to fit the machine's build area
channelMargin = 0 / scaleFactor # 8mm channel margin
tabMargin = 2 / scaleFactor # 2mm tab margin

In [ ]:

import inflation

targetSurf = target_surf
iwv = [isheet.isWallVtx(i) for i in range(isheet.mesh().numVertices())]

In [ ]:
isheet.mesh()

In [ ]:
mesh = isheet.mesh()

In [ ]:
mesh.save("igloo_2D.obj")

In [ ]:
np.save("igloo_is_wall.npy", iwv)

In [ ]:
uv = rparam.uv()

In [ ]:
# !pip install shapely==1.7.0

In [ ]:
import shapely

In [ ]:
import fabrication

In [ ]:
importlib.reload(utils)
importlib.reload(fabrication)
importlib.reload(shapely)
import shapely.geometry as shp


In [ ]:
shapely.__version__

In [ ]:
# new_fabrication.writeFabricationData('fabrication_data/igloo/free_bdry', isheet.mesh(), iwv, targetSurf, uv,
#                                  scale=scaleFactor,
#                                  channelMargin=channelMargin, fuseSeamWidth=None,
#                                  overlap=0.0, smartOuterChannel=True)

In [ ]:
import fabrication
fabrication.writeFabricationData('fabrication_data/igloo_2023_12_17/free_bdry_low_frequency', isheet.mesh(), isheet.mesh(), iwv, targetSurf, uv,
                                 scale=scaleFactor, numTabs=0, inletOffset=0, tabOffset=0.60 / 80,
                                 channelMargin=channelMargin, tabMargin=tabMargin, tabWidth=5, tabHeight=8, fuseSeamWidth=None, inletScale=None,
                                 overlap=0.0, smartOuterChannel=False)

In [ ]:
# import fabrication
# fabrication.writeFabricationData('fabrication_data/igloo/free_bdry', isheet.mesh(), isheet.mesh(), iwv, targetSurf, uv,
#                                  scale=scaleFactor, numTabs=0, inletOffset=0.742, tabOffset=0.60 / 80,
#                                  channelMargin=channelMargin, tabMargin=tabMargin, tabWidth=5, tabHeight=8, fuseSeamWidth=1.0, inletScale=12 / channelMargin / scaleFactor,
#                                  overlap=0.0, smartOuterChannel=True)

In [ ]:
from IPython.display import SVG, display


In [ ]:
display(SVG(filename = "fabrication_data/igloo/free_bdry_low_frequency/orig.wall_boundaries.svg"))